# 数据准备

In [1]:
import os
import numpy as np
import tensorflow as tf

# 读取Shakespeare文本文件
with open('shakespeare.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# 打印文本的前100个字符
print(f"文本长度: {len(text)}")
print(f"文本前100个字符:\n{text[:100]}")

# 创建字符级别的字典
vocab = sorted(set(text))
print(f"字典大小: {len(vocab)}")
print(f"字典内容: {vocab}")

# 创建字符到索引的映射
char_to_idx = {char: idx for idx, char in enumerate(vocab)}
idx_to_char = {idx: char for idx, char in enumerate(vocab)}

print(f"字符到索引的映射:\n{char_to_idx}")

# 打印映射示例
print("\n字符到索引的映射示例:")
for char in text[:20]:
    print(f"'{char}' -> {char_to_idx[char]}")

# 将文本转换为数字序列
text_as_int = np.array([char_to_idx[c] for c in text]) #把全部文本都变为id
print(f"\n文本转换为数字序列的前20个元素:\n{text_as_int[:20]}")
print(f"将数字序列转回字符:\n{''.join([idx_to_char[idx] for idx in text_as_int[:20]])}")


文本长度: 1115394
文本前100个字符:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You
字典大小: 65
字典内容: ['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
字符到索引的映射:
{'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58,

# 把莎士比亚文集分成一个个的样本，每个样本大小为100，每个批次有64个样本

In [2]:
# 定义序列长度和批次大小
import torch
from torch.utils.data import Dataset, DataLoader

seq_length = 100  # 每个样本的序列长度
batch_size = 64   # 每个批次的样本数量

# 创建自定义数据集类
class ShakespeareDataset(Dataset):
    def __init__(self, text_as_int, seq_length):
        self.text_as_int = text_as_int   # 文本的整数表示
        self.seq_length = seq_length     # 序列长度
        self.sub_len = seq_length + 1 #一个样本的长度，+1-预测下一个字符
        
    def __len__(self):
        # 计算可能的序列数量
        # 计算数据集中可以切分出的样本数量
        # 假设文本已经被转换为整数序列 self.text_as_int
        # 每个样本的长度为 seq_length+1（因为要用前seq_length个字符预测下一个字符）
        # 用总长度整除每个样本长度，得到可以切分出的完整样本数
        return len(self.text_as_int) // (self.seq_length + 1)
        
    def __getitem__(self, idx):
        return self.text_as_int[idx*self.sub_len:(idx+1)*self.sub_len]  # idx=0时，返回[0, 101],idx=1时，返回[101, 202]

# 定义collate函数，用于处理批次数据
def collate_fct(batch):
    # 将批次数据转换为张量
    batch = torch.tensor(batch)
    # 输入序列是除了最后一个字符的所有字符
    # 取每个样本的前seq_length个字符作为输入序列（去掉每行的最后一个字符）
    input_batch = batch[:, :-1]
    # 目标序列是除了第一个字符的所有字符
    target_batch = batch[:, 1:]
    return input_batch, target_batch

# 创建数据集实例
shakespeare_dataset = ShakespeareDataset(text_as_int, seq_length)

# 创建数据加载器
dataloader = DataLoader(shakespeare_dataset, batch_size=batch_size, shuffle=True, drop_last=True, collate_fn=collate_fct) # drop_last=True,表示如果数据集大小不能被批量大小整除，则丢弃最后一个批次

# 打印示例，查看输入和目标
for input_batch, target_batch in dataloader:
    print(f"输入批次形状: {input_batch.shape}")
    print(f"目标批次形状: {target_batch.shape}")
    
    # 打印第一个样本的输入和目标
    print(input_batch)
    print(target_batch)
    break

print(f"\n数据集大小: {len(shakespeare_dataset)}")
print(f"批次数量: {len(dataloader)}")


输入批次形状: torch.Size([64, 100])
目标批次形状: torch.Size([64, 100])
tensor([[39, 53, 50,  ..., 63,  1, 46],
        [ 0, 18, 30,  ..., 46, 47, 52],
        [39, 52, 42,  ..., 57,  1, 41],
        ...,
        [56, 58, 12,  ..., 20, 21, 16],
        [ 1, 61, 47,  ...,  1, 53, 44],
        [58,  1, 58,  ..., 59, 56,  1]], dtype=torch.int32)
tensor([[53, 50,  2,  ...,  1, 46, 43],
        [18, 30, 21,  ..., 47, 52, 45],
        [52, 42,  1,  ...,  1, 41, 53],
        ...,
        [58, 12,  0,  ..., 21, 16, 13],
        [61, 47, 58,  ..., 53, 44, 44],
        [ 1, 58, 43,  ..., 56,  1, 39]], dtype=torch.int32)

数据集大小: 11043
批次数量: 172


C:\Users\hsl\AppData\Local\Temp\ipykernel_25272\2961530018.py:29: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:257.)
  batch = torch.tensor(batch)


# 搭建模型

In [4]:
import torch.nn as nn
import torch.nn.functional as F

# 定义RNN模型
class ShakespeareRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, batch_size):
        super(ShakespeareRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim) # 嵌入层
        # RNN层，num_layers表示层数,bidirectional表示是否双向,batch_first表示输入数据的维度顺序，True表示[batch_size, seq_len, feature_dim]，False表示[seq_len, batch_size, feature_dim]
        self.rnn = nn.RNN(
            embedding_dim,
            hidden_dim,
            num_layers=1,
            bidirectional=False,
            batch_first=True
        )
    
        # 定义一个全连接层，将RNN的隐藏状态输出(hidden_dim维)映射到词汇表大小(vocab_size)，用于预测下一个字符的概率分布
        self.dense = nn.Linear(hidden_dim, vocab_size) 
        
    def forward(self, x, hidden=None):  
        # 输入形状: [batch_size, sequence_length]
        x = self.embedding(x)  # 形状: [batch_size, sequence_length, embedding_dim]

        # 调用RNN层进行前向传播，输入x为嵌入后的序列，hidden为可选的初始隐藏状态
        # RNN返回两个值：output为每个时间步的输出（形状为[batch_size, sequence_length, hidden_dim]），
        # hidden为最后一个时间步的隐藏状态（形状为[num_layers, batch_size, hidden_dim]）
        output, hidden = self.rnn(x, hidden)
        output = self.dense(output)  # 形状: [batch_size, sequence_length, vocab_size]
        return output, hidden
    


# 定义模型参数
vocab_size = len(char_to_idx)  # 词汇表大小
embedding_dim = 256  # 嵌入维度
rnn_units = 1024  # RNN单元数量

# 实例化模型
model = ShakespeareRNN(vocab_size, embedding_dim, rnn_units, batch_size)
print(model)


ShakespeareRNN(
  (embedding): Embedding(65, 256)
  (rnn): RNN(256, 1024, batch_first=True)
  (dense): Linear(in_features=1024, out_features=65, bias=True)
)


In [5]:
# 创建一个小批量数据来测试模型
batch_size = 4
seq_length = 100
# 随机生成一个形状为[batch_size, seq_length]的整数张量，数值范围在[0, vocab_size)之间，用作模型的测试输入
test_input = torch.randint(0, vocab_size, (batch_size, seq_length))

# 进行前向计算
with torch.no_grad():
    output, hidden = model(test_input)
    
# 打印输出形状
print(f"输入形状: {test_input.shape}")
print(f"输出形状: {output.shape}")

# 验证输出是否符合预期
assert output.shape == (batch_size, seq_length, vocab_size), "输出形状不符合预期"
assert hidden.shape == (1, batch_size, rnn_units), "隐藏状态形状不符合预期"

print("模型前向计算验证成功！")


输入形状: torch.Size([4, 100])
输出形状: torch.Size([4, 100, 65])
模型前向计算验证成功！
